# SnatchGuard: night-robust perception training (Google Colab)

Only **perception** is trained here: the YOLO26 detector (person / bicycle / motorcycle) and the YOLO26 pose model.
The snatch decision on the edge stays **rule-based**.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).

Everything important (datasets cache, checkpoints, exports) is written to Google Drive, so if Colab disconnects,
just run all cells again: training **resumes** from the last epoch.

| Step | Time on a T4 (approx.) |
|---|---|
| Build datasets | 30–60 min (downloads) |
| Pseudo-label your CCTV (optional, recommended) | 20–40 min |
| Train detector (40 epochs) | 3–5 h |
| Train pose (25 epochs) | 2–3 h |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/snatchguard'
!mkdir -p {DRIVE}

## 1. Get the code
The repository is private. Create a **fine-grained GitHub token** with *read-only Contents* access to the
`snatchguard` repo (GitHub → Settings → Developer settings → Fine-grained tokens) and paste it when asked.
The token is only used for this clone and is not saved in the notebook.

In [ ]:
import getpass, os, subprocess
REPO = 'thilaganiniyavan/snatchguard'   # change if your repo name differs
if not os.path.exists('/content/snatchguard'):
    token = getpass.getpass('GitHub token (read-only): ')
    subprocess.run(['git', 'clone', '--depth', '1', f'https://{token}@github.com/{REPO}.git', '/content/snatchguard'], check=True)
    del token
%cd /content/snatchguard
!git log --oneline -1

In [ ]:
!pip -q install -e ".[edge]" huggingface_hub
import ultralytics; print('ultralytics', ultralytics.__version__)

## 2. Build the datasets
* **ExDark** (real low-light photos), **LLVIP** (real infrared surveillance), **VisDrone** (elevated street views with riders
  and motorcycles), **COCO** street images.
* Daytime images get synthetic **IR / low-light** variants and **CCTV degradation** (blur, heavy JPEG, low resolution).
* Separate `val_day` / `val_night` sets so night accuracy is reported on its own.

Datasets are built on the fast local disk; the download cache lives on Drive so a restart doesn't re-download.

In [ ]:
!python training/prepare_dataset.py --out /content/datasets --cache {DRIVE}/cache \
    --coco-images 12000 --visdrone-images 6000 --llvip-images 6000 --pose-images 15000

## 3. (Recommended) Teach the model your cameras: teacher → student pseudo-labels
A large YOLO26x labels frames from real CCTV recordings; the small edge model then trains on them.
Below we use the **Snatch 1.0** normal videos (real Hyderabad CCTV). `s_2.avi` and `s_3.avi` are held out because the
false-alarm benchmark uses them. Replace with footage from **your own cameras** as soon as you have it: that is the
single biggest accuracy gain.

In [ ]:
!python scripts/snatch_dataset.py get-all --category Normal --max 8 --max-mb 300
!python training/pseudo_label.py --videos data/samples/snatch1.0 --out /content/datasets/sg_det \
    --every 2.0 --max-per-video 300 --exclude s_2.avi s_3.avi --night-aug

## 4. Train the detector (resumes automatically after a disconnect)

In [ ]:
!python training/train.py det --data /content/datasets/sg_det --epochs 40 --imgsz 640 --batch 32 \
    --project {DRIVE}/runs --export-to {DRIVE}/export

## 5. Day vs night accuracy: pretrained vs fine-tuned (put this table in the report)

In [ ]:
!python training/train.py compare --kind det --data /content/datasets/sg_det --imgsz 640 \
    --weights {DRIVE}/runs/det/weights/best.pt --project {DRIVE}/runs
!cat {DRIVE}/runs/compare_det.json

## 6. Train the pose model

In [ ]:
!python training/train.py pose --data /content/datasets/sg_pose --epochs 25 --imgsz 640 --batch 32 \
    --project {DRIVE}/runs --export-to {DRIVE}/export
!python training/train.py compare --kind pose --data /content/datasets/sg_pose --imgsz 640 \
    --weights {DRIVE}/runs/pose/weights/best.pt --project {DRIVE}/runs

## 7. Bring the models to the laptop / edge box
Download `MyDrive/snatchguard/export/det/best.pt` and `export/pose/best.pt`, then on the laptop:

```bash
python scripts/fetch_models.py --from-colab path/to/det/best.pt path/to/pose/best.pt
# then set models.detector / models.pose in configs/edge.yaml to the printed paths
python -m snatchguard.evaluation.benchmark --only snatch_bike    # before/after comparison
```

In [ ]:
!ls -R {DRIVE}/export | head -40